# tool_01 输入文件生成

## 功能
- 扫描根目录下 `.vasp` 结构文件
- 为每个结构创建 `job_{name}/`，生成 **POSCAR、POTCAR、KPOINTS、INCAR**
- 移交检查：四类文件齐全方可提交

## 参考
- `thermol/tool/GO_SEG.py`、`GO_SLURM.py`

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import sys
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
from shared.input_templates import INCAR_TEMPLATE, KPOINTS_TEMPLATE

cfg = load_config()
go_root = Path(cfg.GO_ROOT)

In [ ]:
print(f"📌 扫描结构目录: {go_root}")
vasp_files = sorted(f for f in os.listdir(go_root) if f.endswith(".vasp"))
if not vasp_files:
    raise FileNotFoundError("未找到 .vasp 文件，请将结构放入 GO_ROOT")
print(f"找到 {len(vasp_files)} 个结构:")
for v in vasp_files:
    print("  -", v)

In [ ]:
job_dirs = []
for vasp in vasp_files:
    clean_name = vasp.replace(".vasp", "")
    jobname = f"job_{clean_name}"
    job_path = go_root / jobname
    job_dirs.append(str(job_path))

    print(f"\n>>> 创建 {jobname} ...")
    job_path.mkdir(parents=True, exist_ok=True)

    # POSCAR: 从 .vasp 复制
    shutil.copy(go_root / vasp, job_path / "POSCAR")

    # INCAR, KPOINTS
    (job_path / "INCAR").write_text(INCAR_TEMPLATE)
    (job_path / "KPOINTS").write_text(KPOINTS_TEMPLATE)

    # POTCAR: vaspkit task 103
    print("  运行 vaspkit -task 103 生成 POTCAR ...")
    subprocess.run("vaspkit -task 103", shell=True, cwd=str(job_path), check=False)

print(f"\n✔ 已生成 {len(job_dirs)} 个任务目录")

In [ ]:
# 移交检查：POSCAR、POTCAR、KPOINTS、INCAR 齐全
from shared.pre_check import run_pre_check
if not run_pre_check(str(go_root)):
    raise RuntimeError("移交检查未通过，详见 shared/移交检查.md")